# Real Estate Market Analysis - Task 3
## Age and Price Analysis

### Step 1: Load Merged Dataset

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Set correct path
base_path = Path('/workspaces/real-estate-market-analysis')

# Load merged data from Task 1
df = pd.read_csv(base_path / 'data/processed/merged_real_estate.csv')

print(f"✅ Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

✅ Loaded dataset: 195 rows, 22 columns


### Step 2: Import Age and Price Modules

In [2]:
sys.path.insert(0, str(base_path))

from src.features.age_binner import AgeBinner
from src.features.price_binner import PriceBinner

print("✅ Modules imported successfully")

✅ Modules imported successfully


### Step 3: Calculate Customer Age at Purchase

In [3]:
# Initialize AgeBinner
age_binner = AgeBinner()

# Calculate age
df = age_binner.calculate_age(df)

print(f"\n📊 Age statistics:")
print(f"   Mean age: {df['age'].mean():.1f} years")
print(f"   Median age: {df['age'].median():.1f} years")
print(f"   Min age: {df['age'].min():.0f} years")
print(f"   Max age: {df['age'].max():.0f} years")

✅ Age calculated. Age range: 19 - 76

📊 Age statistics:
   Mean age: 45.6 years
   Median age: 44.0 years
   Min age: 19 years
   Max age: 76 years


### Step 4: Create Age Intervals (10 intervals as specified)

In [4]:
# Create age groups
df = age_binner.create_age_groups(df)

# Show distribution
age_distribution = age_binner.get_age_distribution(df)

✅ Age groups created: 10 groups

📊 Properties sold by age interval:
age_group
19-25     8
25-31    17
31-36    23
36-42    32
42-48    30
48-54    17
54-59    24
59-65    11
65-71    11
71-76     4
Name: count, dtype: int64


In [5]:
# Display age distribution as DataFrame
age_dist_df = df['age_group'].value_counts().sort_index().reset_index()
age_dist_df.columns = ['Age Interval', 'Number of Properties Sold']
print("\n" + age_dist_df.to_string(index=False))


Age Interval  Number of Properties Sold
       19-25                          8
       25-31                         17
       31-36                         23
       36-42                         32
       42-48                         30
       48-54                         17
       54-59                         24
       59-65                         11
       65-71                         11
       71-76                          4


### Step 5: Create Price Intervals (10 bins)

In [6]:
# Initialize PriceBinner
price_binner = PriceBinner(n_bins=10)

# Create price bins
df = price_binner.create_price_bins(df)

# Show distribution
price_distribution = price_binner.get_price_distribution(df)

✅ Price bins created: 10 bins
   Price range: $117,564.07 - $529,317.28

📊 Properties sold by price interval:
price_bin
(117564.069, 197379.85]     20
(197379.85, 206594.532]     19
(206594.532, 219471.88]     20
(219471.88, 229925.43]      19
(229925.43, 243052.59]      20
(243052.59, 256966.376]     19
(256966.376, 291201.74]     19
(291201.74, 327739.248]     20
(327739.248, 389702.618]    19
(389702.618, 529317.28]     20
Name: count, dtype: int64


In [7]:
# Display price bin statistics
price_stats = price_binner.get_price_statistics_by_bin(df)


📊 Price bin statistics:
                          count        min        max       mean
price_bin                                                       
(117564.069, 197379.85]      20  117564.07  197053.51  180843.12
(197379.85, 206594.532]      19  197869.36  206445.42  201987.86
(206594.532, 219471.88]      20  206631.81  219373.41  212532.70
(219471.88, 229925.43]       19  219865.76  229581.78  224716.63
(229925.43, 243052.59]       20  230154.53  243052.59  236996.34
(243052.59, 256966.376]      19  244820.67  256821.64  249605.72
(256966.376, 291201.74]      19  257183.48  290031.26  270433.28
(291201.74, 327739.248]      20  291494.36  326885.34  306808.29
(327739.248, 389702.618]     19  331154.88  388515.14  360638.63
(389702.618, 529317.28]      20  390494.27  529317.28  447793.51


### Step 6: Relationship Between Age and Price

In [8]:
# Calculate covariance and correlation
covariance = df['age'].cov(df['price'])
correlation = df['age'].corr(df['price'])

print("="*60)
print("RELATIONSHIP BETWEEN AGE AND PRICE")
print("="*60)
print(f"\nCovariance: {covariance:.2f}")
print(f"Correlation: {correlation:.4f}")

RELATIONSHIP BETWEEN AGE AND PRICE

Covariance: -178048.05
Correlation: -0.1753


In [9]:
# Group by age group and calculate average price
age_price_analysis = df.groupby('age_group').agg({
    'price': ['mean', 'median', 'count'],
    'age': 'mean'
}).round(2)

age_price_analysis.columns = ['avg_price', 'median_price', 'count', 'avg_age']
print("\n📊 Average Price by Age Group:")
print(age_price_analysis)


📊 Average Price by Age Group:
           avg_price  median_price  count  avg_age
age_group                                         
19-25      311719.74     229426.36      8    23.75
25-31      286550.93     258572.48     17    29.35
31-36      275511.03     252185.99     23    33.83
36-42      287831.22     255507.25     32    39.69
42-48      265492.93     237407.16     30    45.83
48-54      246009.54     234172.39     17    51.41
54-59      267717.77     232483.40     24    56.92
59-65      253495.33     244820.67     11    63.73
65-71      240013.31     224076.84     11    67.91
71-76      268106.20     276537.12      4    73.25


### Step 7: Save Results

In [10]:
# Create reports folder at project root
import os
os.makedirs('../reports', exist_ok=True)

# Save results
age_dist_df.to_csv('../reports/age_distribution.csv', index=False)
price_stats.to_csv('../reports/price_bin_stats.csv')
age_price_analysis.to_csv('../reports/age_price_analysis.csv')

# Save correlation results
correlation_df = pd.DataFrame({
    'Metric': ['Covariance', 'Correlation'],
    'Value': [covariance, correlation]
})
correlation_df.to_csv('../reports/correlation_results.csv', index=False)

print("✅ Results saved to reports/ folder at project root")

✅ Results saved to reports/ folder at project root


### Step 8: Summary Statistics

In [11]:
print("\n" + "="*60)
print("📊 TASK 3 - SUMMARY STATISTICS")
print("="*60)

print("\n👤 AGE STATISTICS:")
print(f"   Most active age group: {age_dist_df.iloc[0]['Age Interval']} ({age_dist_df.iloc[0]['Number of Properties Sold']} purchases)")

print("\n💰 PRICE STATISTICS:")
print(f"   Average price: ${df['price'].mean():,.2f}")
print(f"   Median price: ${df['price'].median():,.2f}")

print("\n📈 AGE-PRICE RELATIONSHIP:")
print(f"   Covariance: {covariance:.2f}")
print(f"   Correlation: {correlation:.4f}")

if correlation > 0:
    print("   → Older customers tend to buy higher-priced properties")
elif correlation < 0:
    print("   → Younger customers tend to buy higher-priced properties")
else:
    print("   → No relationship between age and price")


📊 TASK 3 - SUMMARY STATISTICS

👤 AGE STATISTICS:
   Most active age group: 19-25 (8 purchases)

💰 PRICE STATISTICS:
   Average price: $269,434.56
   Median price: $243,052.59

📈 AGE-PRICE RELATIONSHIP:
   Covariance: -178048.05
   Correlation: -0.1753
   → Younger customers tend to buy higher-priced properties


## ✅ TASK 3 COMPLETE

In [12]:
print("\n" + "="*60)
print("✅ TASK 3 COMPLETED SUCCESSFULLY")
print("="*60)
print("\nCompleted:")
print("  ✓ Calculated customer age at purchase")
print("  ✓ Created 10 age intervals")
print("  ✓ Created 10 price bins")
print("  ✓ Calculated covariance and correlation")
print("\n📁 Results saved to reports/ folder")
print("\n➡️ Ready for Task 4: Data Visualization")


✅ TASK 3 COMPLETED SUCCESSFULLY

Completed:
  ✓ Calculated customer age at purchase
  ✓ Created 10 age intervals
  ✓ Created 10 price bins
  ✓ Calculated covariance and correlation

📁 Results saved to reports/ folder

➡️ Ready for Task 4: Data Visualization
